# Phase 10 — Knowledge Distillation

**Archived Kaggle research notebook.** Paths refer to the original Kaggle environment. Review path variables before execution. Some cleanup cells intentionally remove large temporary artifacts under `/kaggle/working`; run those cells only when the targets have been checked. Large datasets, model weights, adapters, and checkpoints are excluded from this GitHub repository.

In [ ]:
!pip install -q \
    transformers==4.57.3 \
    peft==0.19.1 \
    accelerate==1.14.0 \
    bitsandbytes==0.50.1 \
    codecarbon==3.3.0

In [ ]:
import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Python/PyTorch environment ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

In [ ]:
CATEGORIES = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

NUM_LABELS = len(CATEGORIES)

label_to_idx = {
    label: i
    for i, label in enumerate(CATEGORIES)
}

idx_to_label = {
    i: label
    for label, i in label_to_idx.items()
}

print("Number of classes:", NUM_LABELS)

for label, idx in label_to_idx.items():
    print(idx, "->", label)

In [ ]:
DATA_DIR = "/kaggle/input/datasets/lucky10406/datavj"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "val.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv")
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

In [ ]:
print("\nTraining labels:")
print(
    train_df["class_label"]
    .value_counts()
)

unknown_labels = (
    set(train_df["class_label"].unique())
    - set(CATEGORIES)
)

print("\nUnknown labels:", unknown_labels)

In [ ]:
PHASE9_BACKUP = (
    "/kaggle/input/datasets/lucky10406/"
    "phase9-final-backup"
)

In [ ]:
for root, dirs, files in os.walk(
    "/kaggle/input/datasets/lucky10406"
):
    if "phase9" in root.lower():
        print(root)

        for f in files[:20]:
            print("   ", f)

In [ ]:
import os

ROOT = "/kaggle/input/datasets/lucky10406"

matches = []

for root, dirs, files in os.walk(ROOT):
    for f in files:
        if f == "benchmark_2000.csv":
            full_path = os.path.join(root, f)
            matches.append(full_path)

print("Found:")
for p in matches:
    print(p)

In [ ]:
BENCHMARK_PATH = "/kaggle/input/datasets/lucky10406/phase8-final-backup/benchmark_2000.csv"

benchmark_df = pd.read_csv(BENCHMARK_PATH)

print("Benchmark shape:", benchmark_df.shape)

print(
    benchmark_df["class_label"]
    .value_counts()
)

In [ ]:
!pip uninstall -y transformers
!pip install -q --no-cache-dir transformers==4.57.3

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu keras tf-keras

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

print("TensorFlow disabled for Transformers.")

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

try:
    import tensorflow
    print("TensorFlow still installed:", tensorflow.__version__)
except ImportError:
    print("TensorFlow not installed — good.")

In [ ]:
from transformers.models.distilbert.modeling_distilbert import (
    DistilBertForSequenceClassification
)

from transformers import DistilBertTokenizerFast

print("SUCCESS: DistilBERT imported.")

In [ ]:
import os
import pandas as pd

CATEGORIES = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

NUM_LABELS = len(CATEGORIES)

label_to_idx = {
    label: i
    for i, label in enumerate(CATEGORIES)
}

idx_to_label = {
    i: label
    for label, i in label_to_idx.items()
}

DATA_DIR = "/kaggle/input/datasets/lucky10406/datavj"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "val.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv")
)

BENCHMARK_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

benchmark_df = pd.read_csv(BENCHMARK_PATH)

print("Classes:", NUM_LABELS)
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Benchmark:", benchmark_df.shape)

In [ ]:
STUDENT_NAME = "distilbert-base-uncased"

student_tokenizer = DistilBertTokenizerFast.from_pretrained(
    STUDENT_NAME
)

student_model = DistilBertForSequenceClassification.from_pretrained(
    STUDENT_NAME,
    num_labels=NUM_LABELS,
    id2label=idx_to_label,
    label2id=label_to_idx
)

print("Student loaded:", STUDENT_NAME)
print("Number of labels:", student_model.config.num_labels)

In [ ]:
student_params = sum(
    p.numel()
    for p in student_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{student_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Student size:", round(student_params / 1e6, 2), "M")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        print(root)

In [ ]:
ADAPTER_PATH = "PASTE_THE_FINAL_ADAPTER_PATH_HERE"

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
ADAPTER_PATH = "/kaggle/input/datasets/lucky10406/qwen2-5-3b-instruct-official/kaggle/working/qlora-adapter-final"

In [ ]:
import os

ADAPTER_PATH = "/kaggle/input/datasets/lucky10406/qwen2-5-3b-instruct-official/kaggle/working/qlora-adapter-final"

print("Adapter exists:", os.path.exists(ADAPTER_PATH))
print("Files:")
print(os.listdir(ADAPTER_PATH))

In [ ]:
!pip uninstall -y torchao

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

try:
    import torchao
    print("torchao still installed:", torchao.__version__)
except ImportError:
    print("torchao not installed — good.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

ADAPTER_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "qwen2-5-3b-instruct-official/"
    "kaggle/working/qlora-adapter-final"
)

print("Adapter path exists:", os.path.exists(ADAPTER_PATH))

teacher_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Base model loaded.")

teacher_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("LoRA adapter loaded.")

teacher_model = teacher_model.merge_and_unload()
teacher_model.eval()

print("Teacher loaded and LoRA merged successfully.")

In [ ]:
student_params = 66_961_162

compression_ratio = teacher_params / student_params

print(
    "Teacher / DistilBERT parameter ratio:",
    round(compression_ratio, 2),
    "x"
)

In [ ]:
def format_prompt(tweet):
    return (
        "Classify the following disaster-related tweet into exactly "
        "one of these categories:\n"
        + "\n".join(CATEGORIES)
        + "\n\n"
        f"Tweet: {tweet}\n\n"
        "Answer:"
    )

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def score_candidate_label(
    model,
    tokenizer,
    prompt,
    candidate_label
):
    target_text = " " + candidate_label

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    target_ids = tokenizer(
        target_text,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    input_ids = torch.cat(
        [prompt_ids, target_ids],
        dim=1
    )

    # Important for device_map="auto":
    # input begins on the embedding device.
    input_device = model.get_input_embeddings().weight.device
    input_ids = input_ids.to(input_device)

    outputs = model(
        input_ids=input_ids,
        use_cache=False
    )

    logits = outputs.logits

    prompt_len = prompt_ids.shape[1]
    target_len = target_ids.shape[1]

    start = prompt_len - 1
    end = start + target_len

    target_logits = logits[:, start:end, :]

    log_probs = F.log_softmax(
        target_logits.float(),
        dim=-1
    )

    target_ids_for_logits = target_ids.to(
        target_logits.device
    )

    token_log_probs = log_probs.gather(
        dim=-1,
        index=target_ids_for_logits.unsqueeze(-1)
    ).squeeze(-1)

    # Length-normalized full-label score
    return token_log_probs.mean().item()

In [ ]:
def get_teacher_distribution(
    model,
    tokenizer,
    tweet,
    categories,
    temperature=4.0
):
    prompt = format_prompt(tweet)

    scores = []

    for category in categories:
        score = score_candidate_label(
            model,
            tokenizer,
            prompt,
            category
        )
        scores.append(score)

    scores = torch.tensor(
        scores,
        dtype=torch.float32
    )

    probs = torch.softmax(
        scores / temperature,
        dim=0
    )

    return scores.numpy(), probs.numpy()

In [ ]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/lucky10406/datavj"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "val.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv")
)

BENCHMARK_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

benchmark_df = pd.read_csv(BENCHMARK_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Benchmark:", benchmark_df.shape)

In [ ]:
CATEGORIES = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

In [ ]:
tweet = train_df["text_clean"].iloc[0]
true_label = train_df["class_label"].iloc[0]

scores, probs = get_teacher_distribution(
    teacher_model,
    teacher_tokenizer,
    tweet,
    CATEGORIES,
    temperature=4.0
)

print("Tweet:")
print(tweet)

print("\nTrue label:")
print(true_label)

print("\nTeacher distribution:")

rows = sorted(
    zip(CATEGORIES, scores, probs),
    key=lambda x: x[2],
    reverse=True
)

for label, score, prob in rows:
    print(
        f"{label:45s} "
        f"score={score:8.4f} "
        f"prob={prob:.6f}"
    )

print("\nProbability sum:", probs.sum())
print("Teacher top prediction:", rows[0][0])

In [ ]:
import numpy as np

In [ ]:
for i in range(5):

    tweet = train_df["text_clean"].iloc[i]
    true_label = train_df["class_label"].iloc[i]

    scores, probs = get_teacher_distribution(
        teacher_model,
        teacher_tokenizer,
        tweet,
        CATEGORIES,
        temperature=4.0
    )

    top_idx = int(np.argmax(probs))

    print("=" * 90)
    print("INDEX:", i)
    print("TRUE :", true_label)
    print("TOP  :", CATEGORIES[top_idx])
    print("TOP PROB:", round(float(probs[top_idx]), 4))
    print("SUM:", round(float(probs.sum()), 6))

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PHASE10_DIR = "/kaggle/working/phase10-results"
os.makedirs(PHASE10_DIR, exist_ok=True)

SOFT_LABEL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_train.npy"
)

PROGRESS_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_progress.json"
)

In [ ]:
@torch.no_grad()
def get_teacher_distribution_batched(
    model,
    tokenizer,
    tweet,
    categories,
    temperature=4.0
):
    prompt = format_prompt(tweet)

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    sequences = []
    target_lengths = []

    for category in categories:
        target_ids = tokenizer(
            " " + category,
            add_special_tokens=False
        )["input_ids"]

        sequences.append(
            prompt_ids + target_ids
        )

        target_lengths.append(
            len(target_ids)
        )

    max_len = max(len(x) for x in sequences)

    pad_id = tokenizer.pad_token_id

    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    input_ids = []
    attention_masks = []

    for seq in sequences:
        pad_len = max_len - len(seq)

        input_ids.append(
            seq + [pad_id] * pad_len
        )

        attention_masks.append(
            [1] * len(seq) + [0] * pad_len
        )

    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long
    )

    attention_mask = torch.tensor(
        attention_masks,
        dtype=torch.long
    )

    input_device = (
        model.get_input_embeddings()
        .weight.device
    )

    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(
        input_device
    )

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False
    )

    logits = outputs.logits.float()

    scores = []

    prompt_len = len(prompt_ids)

    for i, target_len in enumerate(
        target_lengths
    ):
        start = prompt_len - 1
        end = start + target_len

        candidate_logits = logits[
            i,
            start:end,
            :
        ]

        target_ids = input_ids[
            i,
            prompt_len:
            prompt_len + target_len
        ]

        log_probs = F.log_softmax(
            candidate_logits,
            dim=-1
        )

        token_log_probs = log_probs.gather(
            dim=-1,
            index=target_ids.unsqueeze(-1)
        ).squeeze(-1)

        scores.append(
            token_log_probs.mean()
        )

    scores = torch.stack(scores)

    probs = torch.softmax(
        scores / temperature,
        dim=0
    )

    return (
        scores.cpu().numpy(),
        probs.cpu().numpy()
    )

In [ ]:
tweet = train_df["text_clean"].iloc[0]

old_scores, old_probs = get_teacher_distribution(
    teacher_model,
    teacher_tokenizer,
    tweet,
    CATEGORIES,
    temperature=4.0
)

new_scores, new_probs = get_teacher_distribution_batched(
    teacher_model,
    teacher_tokenizer,
    tweet,
    CATEGORIES,
    temperature=4.0
)

print("Maximum score difference:")
print(
    np.max(
        np.abs(old_scores - new_scores)
    )
)

print("\nMaximum probability difference:")
print(
    np.max(
        np.abs(old_probs - new_probs)
    )
)

print("\nOld top:",
      CATEGORIES[int(np.argmax(old_probs))])

print("New top:",
      CATEGORIES[int(np.argmax(new_probs))])

In [ ]:
N_TEST = 20

start = time.perf_counter()

for i in range(N_TEST):
    _scores, _probs = (
        get_teacher_distribution_batched(
            teacher_model,
            teacher_tokenizer,
            train_df["text_clean"].iloc[i],
            CATEGORIES,
            temperature=4.0
        )
    )

elapsed = time.perf_counter() - start

per_tweet = elapsed / N_TEST
estimated_hours = (
    per_tweet * len(train_df) / 3600
)

print("20-tweet runtime:", round(elapsed, 2), "s")
print("Seconds per tweet:", round(per_tweet, 3))
print(
    "Estimated full-train runtime:",
    round(estimated_hours, 2),
    "hours"
)

In [ ]:
N = len(train_df)
NUM_CLASSES = len(CATEGORIES)

teacher_soft_labels = np.zeros(
    (N, NUM_CLASSES),
    dtype=np.float32
)

start_idx = 0

# Resume if a partial checkpoint exists
PARTIAL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_partial.npy"
)

if (
    os.path.exists(PARTIAL_PATH)
    and os.path.exists(PROGRESS_PATH)
):
    teacher_soft_labels = np.load(
        PARTIAL_PATH
    )

    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)

    start_idx = progress["next_index"]

    print(
        "Resuming from index:",
        start_idx
    )

else:
    print("Starting from index 0.")

In [ ]:
SAVE_EVERY = 250
TEMPERATURE = 4.0

start_time = time.perf_counter()

for i in range(start_idx, N):

    tweet = train_df[
        "text_clean"
    ].iloc[i]

    _, probs = (
        get_teacher_distribution_batched(
            teacher_model,
            teacher_tokenizer,
            tweet,
            CATEGORIES,
            temperature=TEMPERATURE
        )
    )

    teacher_soft_labels[i] = probs

    if (
        (i + 1) % SAVE_EVERY == 0
        or (i + 1) == N
    ):
        np.save(
            PARTIAL_PATH,
            teacher_soft_labels
        )

        with open(
            PROGRESS_PATH,
            "w"
        ) as f:
            json.dump(
                {
                    "next_index": i + 1,
                    "total": N,
                    "temperature":
                        TEMPERATURE
                },
                f,
                indent=4
            )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        done_this_run = (
            i + 1 - start_idx
        )

        rate = (
            done_this_run / elapsed
        )

        remaining = N - (i + 1)

        eta_min = (
            remaining / rate / 60
            if rate > 0
            else float("inf")
        )

        print(
            f"{i+1}/{N} saved "
            f"| rate={rate:.3f} tweets/s "
            f"| ETA={eta_min:.1f} min"
        )

In [ ]:
np.save(
    SOFT_LABEL_PATH,
    teacher_soft_labels
)

print("Final soft-label matrix saved.")
print("Shape:", teacher_soft_labels.shape)

print(
    "Expected:",
    (len(train_df), len(CATEGORIES))
)

print(
    "NaN count:",
    np.isnan(
        teacher_soft_labels
    ).sum()
)

print(
    "Min row sum:",
    teacher_soft_labels.sum(axis=1).min()
)

print(
    "Max row sum:",
    teacher_soft_labels.sum(axis=1).max()
)

In [2]:
import os
import pandas as pd
import numpy as np

CATEGORIES = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

DATA_DIR = "/kaggle/input/datasets/lucky10406/datavj"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(DATA_DIR, "val.csv")
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv")
)

BENCHMARK_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

benchmark_df = pd.read_csv(BENCHMARK_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Benchmark:", benchmark_df.shape)
print("Classes:", len(CATEGORIES))

Train: (37471, 4)
Val: (8029, 3)
Test: (8030, 3)
Benchmark: (2000, 3)
Classes: 10


In [3]:
import json
import time

PHASE10_DIR = "/kaggle/working/phase10-results"
os.makedirs(PHASE10_DIR, exist_ok=True)

PARTIAL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_partial.npy"
)

PROGRESS_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_progress.json"
)

FINAL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_train.npy"
)

N = len(train_df)
NUM_CLASSES = len(CATEGORIES)

print("Training examples:", N)
print("Classes:", NUM_CLASSES)

Training examples: 37471
Classes: 10


In [4]:
print("teacher_model exists:", "teacher_model" in globals())
print(
    "batched scorer exists:",
    "get_teacher_distribution_batched" in globals()
)

teacher_model exists: False
batched scorer exists: False


In [5]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Environment ready.")
print("PyTorch:", torch.__version__)

Environment ready.
PyTorch: 2.10.0+cu128


In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
import importlib.metadata

try:
    print("torchao:", importlib.metadata.version("torchao"))
except importlib.metadata.PackageNotFoundError:
    print("torchao removed successfully.")

torchao removed successfully.


In [9]:
print("base_model exists:", "base_model" in globals())
print("teacher tokenizer exists:", "teacher_tokenizer" in globals())

base_model exists: True
teacher tokenizer exists: True


In [10]:
from peft import PeftModel

teacher_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("LoRA adapter loaded.")

teacher_model = teacher_model.merge_and_unload()
teacher_model.eval()

print("Teacher loaded and LoRA merged successfully.")

LoRA adapter loaded.
Teacher loaded and LoRA merged successfully.


In [11]:
print("Teacher device map:")
print(teacher_model.hf_device_map)

teacher_params = sum(
    p.numel()
    for p in teacher_model.parameters()
)

print("Teacher parameters:", f"{teacher_params:,}")
print(
    "Teacher parameters (B):",
    round(teacher_params / 1e9, 3)
)

Teacher device map:
{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1}
Teacher parameters: 3,085,938,688
Teacher parameters (B): 3.086


In [12]:
import torch
import torch.nn.functional as F

def format_prompt(tweet):
    return (
        "Classify the following disaster-related tweet into exactly "
        "one of these categories:\n"
        + "\n".join(CATEGORIES)
        + "\n\n"
        f"Tweet: {tweet}\n\n"
        "Answer:"
    )

In [13]:
@torch.no_grad()
def get_teacher_distribution_batched(
    model,
    tokenizer,
    tweet,
    categories,
    temperature=4.0
):
    prompt = format_prompt(tweet)

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    sequences = []
    target_lengths = []

    for category in categories:
        target_ids = tokenizer(
            " " + category,
            add_special_tokens=False
        )["input_ids"]

        sequences.append(prompt_ids + target_ids)
        target_lengths.append(len(target_ids))

    max_len = max(len(x) for x in sequences)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    input_ids = []
    attention_masks = []

    for seq in sequences:
        pad_len = max_len - len(seq)

        input_ids.append(
            seq + [pad_id] * pad_len
        )

        attention_masks.append(
            [1] * len(seq) + [0] * pad_len
        )

    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long
    )

    attention_mask = torch.tensor(
        attention_masks,
        dtype=torch.long
    )

    input_device = (
        model.get_input_embeddings().weight.device
    )

    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(input_device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False
    )

    logits = outputs.logits.float()

    scores = []

    prompt_len = len(prompt_ids)

    for i, target_len in enumerate(target_lengths):
        start = prompt_len - 1
        end = start + target_len

        candidate_logits = logits[
            i,
            start:end,
            :
        ]

        target_ids = input_ids[
            i,
            prompt_len:
            prompt_len + target_len
        ]

        log_probs = F.log_softmax(
            candidate_logits,
            dim=-1
        )

        token_log_probs = log_probs.gather(
            dim=-1,
            index=target_ids.unsqueeze(-1)
        ).squeeze(-1)

        scores.append(
            token_log_probs.mean()
        )

    scores = torch.stack(scores)

    probs = torch.softmax(
        scores / temperature,
        dim=0
    )

    return (
        scores.cpu().numpy(),
        probs.cpu().numpy()
    )

In [14]:
print(
    "batched scorer exists:",
    "get_teacher_distribution_batched" in globals()
)

batched scorer exists: True


In [15]:
import numpy as np

tweet = train_df["text_clean"].iloc[0]

scores, probs = get_teacher_distribution_batched(
    teacher_model,
    teacher_tokenizer,
    tweet,
    CATEGORIES,
    temperature=4.0
)

print("Top class:", CATEGORIES[int(np.argmax(probs))])
print("Probability sum:", probs.sum())
print("NaNs:", np.isnan(probs).sum())

Top class: rescue_volunteering_or_donation_effort
Probability sum: 0.99999994
NaNs: 0


In [16]:
import os
import json
import time
import numpy as np

PHASE10_DIR = "/kaggle/working/phase10-results"
os.makedirs(PHASE10_DIR, exist_ok=True)

PARTIAL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_partial.npy"
)

PROGRESS_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_progress.json"
)

FINAL_PATH = os.path.join(
    PHASE10_DIR,
    "teacher_soft_labels_train.npy"
)

N = len(train_df)
NUM_CLASSES = len(CATEGORIES)

print("Training examples:", N)
print("Classes:", NUM_CLASSES)

Training examples: 37471
Classes: 10


In [17]:
if (
    os.path.exists(PARTIAL_PATH)
    and os.path.exists(PROGRESS_PATH)
):
    teacher_soft_labels = np.load(PARTIAL_PATH)

    with open(PROGRESS_PATH, "r") as f:
        progress = json.load(f)

    start_idx = int(progress["next_index"])

    print("RESUMING from index:", start_idx)

else:
    teacher_soft_labels = np.zeros(
        (N, NUM_CLASSES),
        dtype=np.float32
    )

    start_idx = 0

    print("STARTING from index 0.")

STARTING from index 0.


In [18]:
SAVE_EVERY = 250
TEMPERATURE = 4.0

start_time = time.perf_counter()

for i in range(start_idx, N):

    tweet = train_df["text_clean"].iloc[i]

    _, probs = get_teacher_distribution_batched(
        teacher_model,
        teacher_tokenizer,
        tweet,
        CATEGORIES,
        temperature=TEMPERATURE
    )

    teacher_soft_labels[i] = probs

    if (
        (i + 1) % SAVE_EVERY == 0
        or (i + 1) == N
    ):
        np.save(
            PARTIAL_PATH,
            teacher_soft_labels
        )

        with open(PROGRESS_PATH, "w") as f:
            json.dump(
                {
                    "next_index": i + 1,
                    "total": N,
                    "temperature": TEMPERATURE
                },
                f,
                indent=4
            )

        elapsed = time.perf_counter() - start_time
        done_this_run = i + 1 - start_idx

        rate = done_this_run / elapsed
        remaining = N - (i + 1)

        eta_min = (
            remaining / rate / 60
            if rate > 0
            else float("inf")
        )

        print(
            f"{i+1}/{N} saved "
            f"| rate={rate:.3f} tweets/s "
            f"| ETA={eta_min:.1f} min"
        )

250/37471 saved | rate=2.561 tweets/s | ETA=242.3 min
500/37471 saved | rate=2.469 tweets/s | ETA=249.5 min
750/37471 saved | rate=2.444 tweets/s | ETA=250.4 min
1000/37471 saved | rate=2.413 tweets/s | ETA=252.0 min
1250/37471 saved | rate=2.385 tweets/s | ETA=253.1 min
1500/37471 saved | rate=2.366 tweets/s | ETA=253.4 min
1750/37471 saved | rate=2.356 tweets/s | ETA=252.7 min
2000/37471 saved | rate=2.356 tweets/s | ETA=250.9 min
2250/37471 saved | rate=2.354 tweets/s | ETA=249.3 min
2500/37471 saved | rate=2.359 tweets/s | ETA=247.1 min
2750/37471 saved | rate=2.357 tweets/s | ETA=245.5 min
3000/37471 saved | rate=2.357 tweets/s | ETA=243.8 min
3250/37471 saved | rate=2.356 tweets/s | ETA=242.0 min
3500/37471 saved | rate=2.358 tweets/s | ETA=240.1 min
3750/37471 saved | rate=2.359 tweets/s | ETA=238.2 min
4000/37471 saved | rate=2.360 tweets/s | ETA=236.4 min
4250/37471 saved | rate=2.361 tweets/s | ETA=234.5 min
4500/37471 saved | rate=2.362 tweets/s | ETA=232.6 min
4750/37471 sa

In [19]:
np.save(
    FINAL_PATH,
    teacher_soft_labels
)

print("Final teacher soft labels saved.")
print("Shape:", teacher_soft_labels.shape)
print("NaN count:", np.isnan(teacher_soft_labels).sum())

row_sums = teacher_soft_labels.sum(axis=1)

print("Min row sum:", row_sums.min())
print("Max row sum:", row_sums.max())
print("Mean row sum:", row_sums.mean())

Final teacher soft labels saved.
Shape: (37471, 10)
NaN count: 0
Min row sum: 0.9999998
Max row sum: 1.0000002
Mean row sum: 1.0


In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch.nn.functional as F
import numpy as np

In [21]:
label_to_idx = {
    label: i
    for i, label in enumerate(CATEGORIES)
}

train_label_ids = [
    label_to_idx[x]
    for x in train_df["class_label"].tolist()
]

val_label_ids = [
    label_to_idx[x]
    for x in val_df["class_label"].tolist()
]

In [22]:
class DistillationDataset(Dataset):
    def __init__(
        self,
        texts,
        hard_labels,
        soft_labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.hard_labels = hard_labels
        self.soft_labels = soft_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids":
                enc["input_ids"].squeeze(0),

            "attention_mask":
                enc["attention_mask"].squeeze(0),

            "hard_label":
                torch.tensor(
                    self.hard_labels[idx],
                    dtype=torch.long
                ),

            "soft_label":
                torch.tensor(
                    self.soft_labels[idx],
                    dtype=torch.float32
                )
        }

In [24]:
from transformers import DistilBertTokenizerFast

STUDENT_NAME = "distilbert-base-uncased"

student_tokenizer = DistilBertTokenizerFast.from_pretrained(
    STUDENT_NAME
)

print("Student tokenizer loaded.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Student tokenizer loaded.


In [25]:
train_dataset = DistillationDataset(
    train_df["text_clean"].tolist(),
    train_label_ids,
    teacher_soft_labels,
    student_tokenizer,
    max_length=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Training examples:", len(train_dataset))
print("Training batches:", len(train_loader))

Training examples: 37471
Training batches: 2342


In [26]:
print("train_df:", "train_df" in globals())
print("teacher_soft_labels:", "teacher_soft_labels" in globals())
print("train_label_ids:", "train_label_ids" in globals())
print("student_tokenizer:", "student_tokenizer" in globals())
print("DistillationDataset:", "DistillationDataset" in globals())

train_df: True
teacher_soft_labels: True
train_label_ids: True
student_tokenizer: True
DistillationDataset: True


In [27]:
import numpy as np

teacher_soft_labels = np.load(
    "/kaggle/working/phase10-results/teacher_soft_labels_train.npy"
)

print("Soft labels:", teacher_soft_labels.shape)

Soft labels: (37471, 10)


In [28]:
from transformers import DistilBertTokenizerFast
from torch.utils.data import DataLoader

STUDENT_NAME = "distilbert-base-uncased"

student_tokenizer = DistilBertTokenizerFast.from_pretrained(
    STUDENT_NAME
)

train_dataset = DistillationDataset(
    train_df["text_clean"].tolist(),
    train_label_ids,
    teacher_soft_labels,
    student_tokenizer,
    max_length=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Training examples:", len(train_dataset))
print("Training batches:", len(train_loader))

Training examples: 37471
Training batches: 2342


In [29]:
import torch
import torch.nn.functional as F

def distillation_loss(
    student_logits,
    teacher_soft_labels,
    true_labels,
    temperature=4.0,
    alpha=0.5
):
    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=-1
    )

    soft_loss = F.kl_div(
        student_log_probs,
        teacher_soft_labels,
        reduction="batchmean"
    ) * (temperature ** 2)

    hard_loss = F.cross_entropy(
        student_logits,
        true_labels
    )

    total_loss = (
        alpha * soft_loss
        + (1 - alpha) * hard_loss
    )

    return total_loss, soft_loss, hard_loss


dummy_student_logits = torch.randn(4, 10)

dummy_teacher_soft = torch.tensor(
    teacher_soft_labels[:4],
    dtype=torch.float32
)

dummy_true = torch.tensor(
    train_label_ids[:4],
    dtype=torch.long
)

loss, soft_loss, hard_loss = distillation_loss(
    dummy_student_logits,
    dummy_teacher_soft,
    dummy_true
)

print("Total loss:", loss.item())
print("Soft loss :", soft_loss.item())
print("Hard loss :", hard_loss.item())

Total loss: 1.7670365571975708
Soft loss : 0.9419640898704529
Hard loss : 2.592108964920044


In [31]:
label_to_idx = {
    label: i
    for i, label in enumerate(CATEGORIES)
}

idx_to_label = {
    i: label
    for label, i in label_to_idx.items()
}

print(label_to_idx)
print(idx_to_label)

{'injured_or_dead_people': 0, 'rescue_volunteering_or_donation_effort': 1, 'sympathy_and_support': 2, 'infrastructure_and_utility_damage': 3, 'not_humanitarian': 4, 'caution_and_advice': 5, 'displaced_people_and_evacuations': 6, 'requests_or_urgent_needs': 7, 'missing_or_found_people': 8, 'other_relevant_information': 9}
{0: 'injured_or_dead_people', 1: 'rescue_volunteering_or_donation_effort', 2: 'sympathy_and_support', 3: 'infrastructure_and_utility_damage', 4: 'not_humanitarian', 5: 'caution_and_advice', 6: 'displaced_people_and_evacuations', 7: 'requests_or_urgent_needs', 8: 'missing_or_found_people', 9: 'other_relevant_information'}


In [32]:
from transformers import DistilBertForSequenceClassification
import torch

student_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=10,
    id2label=idx_to_label,
    label2id=label_to_idx
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

student_model = student_model.to(device)

print("Student device:", device)
print("Labels:", student_model.config.num_labels)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Student device: cuda
Labels: 10


In [33]:
from torch.optim import AdamW

optimizer = AdamW(
    student_model.parameters(),
    lr=2e-5
)

print("Optimizer ready.")

Optimizer ready.


In [34]:
checks = [
    "train_loader",
    "teacher_soft_labels",
    "train_label_ids",
    "student_tokenizer",
    "distillation_loss",
]

for name in checks:
    print(name, ":", name in globals())

train_loader : True
teacher_soft_labels : True
train_label_ids : True
student_tokenizer : True
distillation_loss : True


In [35]:
EPOCHS = 3
TEMPERATURE = 4.0
ALPHA = 0.5

student_model.train()

for epoch in range(EPOCHS):

    total_loss_epoch = 0.0
    total_soft_epoch = 0.0
    total_hard_epoch = 0.0

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        hard_labels = batch["hard_label"].to(device)
        soft_labels = batch["soft_label"].to(device)

        optimizer.zero_grad()

        outputs = student_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss, soft_loss, hard_loss = distillation_loss(
            outputs.logits,
            soft_labels,
            hard_labels,
            temperature=TEMPERATURE,
            alpha=ALPHA
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss_epoch += loss.item()
        total_soft_epoch += soft_loss.item()
        total_hard_epoch += hard_loss.item()

        if (step + 1) % 250 == 0:
            print(
                f"Epoch {epoch+1} "
                f"| Step {step+1}/{len(train_loader)} "
                f"| Loss {loss.item():.4f}"
            )

    avg_total = total_loss_epoch / len(train_loader)
    avg_soft = total_soft_epoch / len(train_loader)
    avg_hard = total_hard_epoch / len(train_loader)

    print("\n" + "=" * 70)
    print(f"EPOCH {epoch+1} COMPLETE")
    print("Avg total loss:", avg_total)
    print("Avg soft loss :", avg_soft)
    print("Avg hard loss :", avg_hard)
    print("=" * 70 + "\n")

Epoch 1 | Step 250/2342 | Loss 0.5522
Epoch 1 | Step 500/2342 | Loss 0.6213
Epoch 1 | Step 750/2342 | Loss 0.5706
Epoch 1 | Step 1000/2342 | Loss 0.6192
Epoch 1 | Step 1250/2342 | Loss 0.6056
Epoch 1 | Step 1500/2342 | Loss 0.5342
Epoch 1 | Step 1750/2342 | Loss 0.6374
Epoch 1 | Step 2000/2342 | Loss 0.4237
Epoch 1 | Step 2250/2342 | Loss 0.6746

EPOCH 1 COMPLETE
Avg total loss: 0.5517928843107313
Avg soft loss : 0.1875727581708071
Avg hard loss : 0.916013009928924

Epoch 2 | Step 250/2342 | Loss 0.3956
Epoch 2 | Step 500/2342 | Loss 0.4615
Epoch 2 | Step 750/2342 | Loss 0.4240
Epoch 2 | Step 1000/2342 | Loss 0.4340
Epoch 2 | Step 1250/2342 | Loss 0.5276
Epoch 2 | Step 1500/2342 | Loss 0.4449
Epoch 2 | Step 1750/2342 | Loss 0.3815
Epoch 2 | Step 2000/2342 | Loss 0.4361
Epoch 2 | Step 2250/2342 | Loss 0.5135

EPOCH 2 COMPLETE
Avg total loss: 0.460145332014632
Avg soft loss : 0.1895718117881595
Avg hard loss : 0.7307188517257355

Epoch 3 | Step 250/2342 | Loss 0.4705
Epoch 3 | Step 500/2

In [36]:
import os
import json

PHASE10_DIR = "/kaggle/working/phase10-results"
os.makedirs(PHASE10_DIR, exist_ok=True)

DISTILLED_PATH = "/kaggle/working/distilled-student"

student_model.save_pretrained(
    DISTILLED_PATH,
    safe_serialization=True
)

student_tokenizer.save_pretrained(
    DISTILLED_PATH
)

training_summary = {
    "student_model": "distilbert-base-uncased",
    "teacher_model": "Qwen2.5-3B-Instruct + merged QLoRA adapter",
    "teacher_parameters": 3085938688,
    "student_parameters": 66961162,
    "num_classes": 10,
    "epochs": 3,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "temperature": 4.0,
    "alpha": 0.5,
    "max_length": 128,
    "training_examples": len(train_df),

    "epoch_1": {
        "avg_total_loss": 0.5517928843107313,
        "avg_soft_loss": 0.1875727581708071,
        "avg_hard_loss": 0.916013009928924
    },

    "epoch_2": {
        "avg_total_loss": 0.460145332014632,
        "avg_soft_loss": 0.1895718117881595,
        "avg_hard_loss": 0.7307188517257355
    },

    "epoch_3": {
        "avg_total_loss": 0.41307219714053567,
        "avg_soft_loss": 0.206461010817327,
        "avg_hard_loss": 0.6196833830120012
    },

    "teacher_soft_label_method":
        "length-normalized full-label conditional sequence scoring"
}

with open(
    os.path.join(
        PHASE10_DIR,
        "distillation_training_summary.json"
    ),
    "w"
) as f:
    json.dump(training_summary, f, indent=4)

print("Distilled student saved.")
print(DISTILLED_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Distilled student saved.
/kaggle/working/distilled-student


In [37]:
import torch
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

student_model.eval()

def predict_student(text):

    enc = student_tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding=False,
        return_tensors="pt"
    )

    enc = {
        k: v.to(device)
        for k, v in enc.items()
    }

    with torch.no_grad():

        outputs = student_model(**enc)

        pred_idx = (
            outputs.logits
            .argmax(dim=-1)
            .item()
        )

    return CATEGORIES[pred_idx]

In [38]:
student_predictions = []

start = time.perf_counter()

for i, row in benchmark_df.iterrows():

    pred = predict_student(
        row["text_clean"]
    )

    student_predictions.append(pred)

    if (i + 1) % 200 == 0:
        print(
            f"{i+1}/{len(benchmark_df)}"
        )

total_time_student = (
    time.perf_counter() - start
)

print(
    "Total benchmark time:",
    total_time_student
)

200/2000
400/2000
600/2000
800/2000
1000/2000
1200/2000
1400/2000
1600/2000
1800/2000
2000/2000
Total benchmark time: 9.061206254002172


In [39]:
y_true = benchmark_df[
    "class_label"
].tolist()

y_pred = student_predictions

student_results = {
    "model": "distilled_student",
    "n_samples": len(y_true),

    "accuracy": accuracy_score(
        y_true,
        y_pred
    ),

    "macro_precision": precision_score(
        y_true,
        y_pred,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true,
        y_pred,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true,
        y_pred,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "weighted_f1": f1_score(
        y_true,
        y_pred,
        labels=CATEGORIES,
        average="weighted",
        zero_division=0
    ),

    "total_inference_time_s":
        total_time_student,

    "samples_per_second":
        len(y_true) / total_time_student,

    "unmatched_count": 0
}

print("=" * 70)
print("DISTILLED STUDENT — 2000 EXAMPLES")
print("=" * 70)

for k, v in student_results.items():
    print(k, ":", v)

DISTILLED STUDENT — 2000 EXAMPLES
model : distilled_student
n_samples : 2000
accuracy : 0.7735
macro_precision : 0.7600140069690334
macro_recall : 0.7680111060543814
macro_f1 : 0.7629680730982821
weighted_f1 : 0.7715770921582532
total_inference_time_s : 9.061206254002172
samples_per_second : 220.7211649240007
unmatched_count : 0


In [40]:
with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_2000_results.json"
    ),
    "w"
) as f:
    json.dump(
        student_results,
        f,
        indent=4
    )


pred_df = benchmark_df.copy()

pred_df[
    "predicted_distilled"
] = student_predictions

pred_df[
    "correct_distilled"
] = (
    pred_df["class_label"]
    ==
    pred_df["predicted_distilled"]
)

pred_df.to_csv(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_2000_predictions.csv"
    ),
    index=False
)

In [41]:
report = classification_report(
    y_true,
    y_pred,
    labels=CATEGORIES,
    target_names=CATEGORIES,
    zero_division=0
)

print(report)

with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_classification_report.txt"
    ),
    "w"
) as f:
    f.write(report)


cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CATEGORIES
)

pd.DataFrame(
    cm,
    index=CATEGORIES,
    columns=CATEGORIES
).to_csv(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_confusion_matrix.csv"
    )
)

                                        precision    recall  f1-score   support

                injured_or_dead_people       0.93      0.93      0.93       191
rescue_volunteering_or_donation_effort       0.83      0.87      0.85       556
                  sympathy_and_support       0.87      0.84      0.85       234
     infrastructure_and_utility_damage       0.83      0.86      0.85       214
                      not_humanitarian       0.58      0.67      0.62       165
                    caution_and_advice       0.77      0.70      0.73       141
      displaced_people_and_evacuations       0.86      0.87      0.86       105
              requests_or_urgent_needs       0.55      0.56      0.55        68
               missing_or_found_people       0.80      0.89      0.84         9
            other_relevant_information       0.57      0.50      0.54       317

                              accuracy                           0.77      2000
                             macro avg

In [42]:
print("EXACT DISTILLED STUDENT RESULTS")
print("=" * 70)

for k, v in student_results.items():
    print(k, ":", v)

EXACT DISTILLED STUDENT RESULTS
model : distilled_student
n_samples : 2000
accuracy : 0.7735
macro_precision : 0.7600140069690334
macro_recall : 0.7680111060543814
macro_f1 : 0.7629680730982821
weighted_f1 : 0.7715770921582532
total_inference_time_s : 9.061206254002172
samples_per_second : 220.7211649240007
unmatched_count : 0


In [43]:
# ============================================================
# PHASE 10 — HARD-LABEL-ONLY ABLATION
# ============================================================

import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

hard_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=10,
    id2label=idx_to_label,
    label2id=label_to_idx
)

hard_model = hard_model.to(device)

print("Hard-label control model ready.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hard-label control model ready.


In [44]:
from torch.optim import AdamW

hard_optimizer = AdamW(
    hard_model.parameters(),
    lr=2e-5
)

In [45]:
EPOCHS = 3

hard_model.train()

hard_training_history = []

for epoch in range(EPOCHS):

    total_loss = 0.0

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["hard_label"].to(device)

        hard_optimizer.zero_grad()

        outputs = hard_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = F.cross_entropy(
            outputs.logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            hard_model.parameters(),
            max_norm=1.0
        )

        hard_optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 250 == 0:
            print(
                f"Epoch {epoch+1} "
                f"| Step {step+1}/{len(train_loader)} "
                f"| Loss {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    hard_training_history.append(
        {
            "epoch": epoch + 1,
            "avg_loss": avg_loss
        }
    )

    print("\n" + "=" * 70)
    print(f"HARD-LABEL EPOCH {epoch+1} COMPLETE")
    print("Avg loss:", avg_loss)
    print("=" * 70 + "\n")

Epoch 1 | Step 250/2342 | Loss 1.0933
Epoch 1 | Step 500/2342 | Loss 1.4306
Epoch 1 | Step 750/2342 | Loss 1.2216
Epoch 1 | Step 1000/2342 | Loss 0.3461
Epoch 1 | Step 1250/2342 | Loss 0.8014
Epoch 1 | Step 1500/2342 | Loss 0.4286
Epoch 1 | Step 1750/2342 | Loss 0.9924
Epoch 1 | Step 2000/2342 | Loss 0.7706
Epoch 1 | Step 2250/2342 | Loss 0.7529

HARD-LABEL EPOCH 1 COMPLETE
Avg loss: 0.8092756447160376

Epoch 2 | Step 250/2342 | Loss 0.5815
Epoch 2 | Step 500/2342 | Loss 0.2626
Epoch 2 | Step 750/2342 | Loss 0.7875
Epoch 2 | Step 1000/2342 | Loss 0.4598
Epoch 2 | Step 1250/2342 | Loss 0.6889
Epoch 2 | Step 1500/2342 | Loss 0.3444
Epoch 2 | Step 1750/2342 | Loss 0.3377
Epoch 2 | Step 2000/2342 | Loss 0.5375
Epoch 2 | Step 2250/2342 | Loss 0.3558

HARD-LABEL EPOCH 2 COMPLETE
Avg loss: 0.5658007350000278

Epoch 3 | Step 250/2342 | Loss 0.3521
Epoch 3 | Step 500/2342 | Loss 0.4959
Epoch 3 | Step 750/2342 | Loss 1.0409
Epoch 3 | Step 1000/2342 | Loss 1.1597
Epoch 3 | Step 1250/2342 | Loss 0

In [46]:
HARD_PATH = "/kaggle/working/hard-label-student"

hard_model.save_pretrained(
    HARD_PATH,
    safe_serialization=True
)

student_tokenizer.save_pretrained(
    HARD_PATH
)

print("Hard-label student saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Hard-label student saved.


In [47]:
hard_model.eval()

def predict_hard_student(text):

    enc = student_tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding=False,
        return_tensors="pt"
    )

    enc = {
        k: v.to(device)
        for k, v in enc.items()
    }

    with torch.no_grad():
        logits = hard_model(**enc).logits

    pred_idx = logits.argmax(
        dim=-1
    ).item()

    return CATEGORIES[pred_idx]

In [48]:
import time

hard_predictions = []

start = time.perf_counter()

for i, row in benchmark_df.iterrows():

    pred = predict_hard_student(
        row["text_clean"]
    )

    hard_predictions.append(pred)

    if (i + 1) % 200 == 0:
        print(f"{i+1}/2000")

hard_total_time = (
    time.perf_counter() - start
)

200/2000
400/2000
600/2000
800/2000
1000/2000
1200/2000
1400/2000
1600/2000
1800/2000
2000/2000


In [49]:
hard_results = {
    "model": "hard_label_student",
    "n_samples": 2000,

    "accuracy": accuracy_score(
        y_true,
        hard_predictions
    ),

    "macro_precision": precision_score(
        y_true,
        hard_predictions,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true,
        hard_predictions,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true,
        hard_predictions,
        labels=CATEGORIES,
        average="macro",
        zero_division=0
    ),

    "weighted_f1": f1_score(
        y_true,
        hard_predictions,
        labels=CATEGORIES,
        average="weighted",
        zero_division=0
    ),

    "total_inference_time_s":
        hard_total_time,

    "samples_per_second":
        2000 / hard_total_time,

    "unmatched_count": 0
}

print("=" * 70)
print("HARD-LABEL STUDENT — EXACT RESULTS")
print("=" * 70)

for k, v in hard_results.items():
    print(k, ":", v)

HARD-LABEL STUDENT — EXACT RESULTS
model : hard_label_student
n_samples : 2000
accuracy : 0.764
macro_precision : 0.7538058234843132
macro_recall : 0.7479424017301597
macro_f1 : 0.7466958537197546
weighted_f1 : 0.7551154514211491
total_inference_time_s : 8.951577346000704
samples_per_second : 223.4243109001946
unmatched_count : 0


In [50]:
with open(
    os.path.join(
        PHASE10_DIR,
        "hard_label_student_2000_results.json"
    ),
    "w"
) as f:
    json.dump(
        hard_results,
        f,
        indent=4
    )

In [51]:
# ============================================================
# PHASE 10 QUICK BACKUP ZIP
# ============================================================

import os
import shutil

PHASE10_DIR = "/kaggle/working/phase10-results"
BACKUP_DIR = "/kaggle/working/phase10-backup"

if os.path.exists(BACKUP_DIR):
    shutil.rmtree(BACKUP_DIR)

os.makedirs(BACKUP_DIR, exist_ok=True)

# Copy Phase 10 results
if os.path.exists(PHASE10_DIR):
    shutil.copytree(
        PHASE10_DIR,
        os.path.join(BACKUP_DIR, "phase10-results")
    )

# Copy distilled model
DISTILLED_PATH = "/kaggle/working/distilled-student"

if os.path.exists(DISTILLED_PATH):
    shutil.copytree(
        DISTILLED_PATH,
        os.path.join(BACKUP_DIR, "distilled-student")
    )

# Copy hard-label ablation model
HARD_PATH = "/kaggle/working/hard-label-student"

if os.path.exists(HARD_PATH):
    shutil.copytree(
        HARD_PATH,
        os.path.join(BACKUP_DIR, "hard-label-student")
    )

# Copy exact benchmark
BENCHMARK_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

if os.path.exists(BENCHMARK_PATH):
    shutil.copy(
        BENCHMARK_PATH,
        os.path.join(BACKUP_DIR, "benchmark_2000.csv")
    )

# README
readme = """
PHASE 10 — KNOWLEDGE DISTILLATION
================================

Teacher:
Qwen2.5-3B-Instruct + merged QLoRA
Parameters: 3,085,938,688

Student:
DistilBERT
Parameters: 66,961,162
Approx teacher/student parameter ratio: 46.08x

Classes:
10

Training examples:
37,471

Distillation:
Temperature = 4.0
Alpha = 0.5
Epochs = 3
Batch size = 16
Learning rate = 2e-5
Max length = 128

Teacher soft-label method:
Length-normalized full-label conditional sequence scoring.

DISTILLED STUDENT — 2000 BENCHMARK
Accuracy = 0.7735
Macro Precision = 0.7600140069690334
Macro Recall = 0.7680111060543814
Macro F1 = 0.7629680730982821
Weighted F1 = 0.7715770921582532
Total inference time = 9.061206254002172 s
Throughput = 220.7211649240007 samples/s
Unmatched = 0

HARD-LABEL DISTILBERT — 2000 BENCHMARK
Accuracy = 0.764
Macro Precision = 0.7538058234843132
Macro Recall = 0.7479424017301597
Macro F1 = 0.7466958537197546
Weighted F1 = 0.7551154514211491
Total inference time = 8.951577346000704 s
Throughput = 223.4243109001946 samples/s
Unmatched = 0

DISTILLATION BENEFIT
Accuracy gain = +0.95 percentage points
Macro F1 gain = +1.63 percentage points
Weighted F1 gain = +1.65 percentage points

Remaining Phase 10 measurements:
- formal latency
- GPU memory
- checkpoint size
- energy / CO2

These can be added before Phase 11 final synthesis.
"""

with open(
    os.path.join(BACKUP_DIR, "README_PHASE10.txt"),
    "w"
) as f:
    f.write(readme)

ZIP_BASE = "/kaggle/working/phase10-final-backup"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    BACKUP_DIR
)

print("=" * 70)
print("PHASE 10 BACKUP CREATED")
print("=" * 70)
print(ZIP_BASE + ".zip")

PHASE 10 BACKUP CREATED
/kaggle/working/phase10-final-backup.zip


In [52]:
import os
import gc
import json
import time
import numpy as np
import pandas as pd
import torch

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

PHASE10_DIR = "/kaggle/working/phase10-results"
DISTILLED_PATH = "/kaggle/working/distilled-student"

device = torch.device(
    "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

# Remove other models if still in memory
for name in ["hard_model", "student_model", "teacher_model", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

student_tokenizer = DistilBertTokenizerFast.from_pretrained(
    DISTILLED_PATH
)

student_model = (
    DistilBertForSequenceClassification
    .from_pretrained(DISTILLED_PATH)
    .to(device)
)

student_model.eval()

print("Distilled student loaded.")
print("Device:", device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Distilled student loaded.
Device: cuda:0


In [53]:
@torch.no_grad()
def predict_distilled(text):

    enc = student_tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding=False,
        return_tensors="pt"
    )

    enc = {
        k: v.to(device)
        for k, v in enc.items()
    }

    logits = student_model(**enc).logits

    pred_idx = logits.argmax(
        dim=-1
    ).item()

    return CATEGORIES[pred_idx]

In [54]:
LATENCY_TEXT = benchmark_df["text_clean"].iloc[0]

# Warmup
for _ in range(5):
    _ = predict_distilled(LATENCY_TEXT)

times = []

for _ in range(50):

    torch.cuda.synchronize()

    start = time.perf_counter()

    _ = predict_distilled(LATENCY_TEXT)

    torch.cuda.synchronize()

    times.append(
        time.perf_counter() - start
    )

distilled_runtime = {
    "model": "distilled_student",
    "warmup_runs": 5,
    "trials": 50,
    "mean_latency_s": float(np.mean(times)),
    "median_latency_s": float(np.median(times)),
    "std_latency_s": float(np.std(times)),
    "min_latency_s": float(np.min(times)),
    "max_latency_s": float(np.max(times)),
    "throughput_samples_per_s":
        float(1 / np.mean(times))
}

print("=" * 70)
print("DISTILLED STUDENT — LATENCY")
print("=" * 70)

for k, v in distilled_runtime.items():
    print(k, ":", v)

with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_runtime.json"
    ),
    "w"
) as f:
    json.dump(
        distilled_runtime,
        f,
        indent=4
    )

DISTILLED STUDENT — LATENCY
model : distilled_student
warmup_runs : 5
trials : 50
mean_latency_s : 0.004398239360307343
median_latency_s : 0.004378193500087946
std_latency_s : 0.00015971987487546331
min_latency_s : 0.004146933999436442
max_latency_s : 0.004868238000199199
throughput_samples_per_s : 227.36370581025434


In [55]:
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict_distilled(LATENCY_TEXT)

distilled_memory = {
    "model": "distilled_student",
    "gpus": {}
}

for i in range(torch.cuda.device_count()):

    distilled_memory["gpus"][f"gpu_{i}"] = {
        "allocated_gb":
            torch.cuda.memory_allocated(i) / 1024**3,

        "reserved_gb":
            torch.cuda.memory_reserved(i) / 1024**3,

        "max_allocated_gb":
            torch.cuda.max_memory_allocated(i) / 1024**3,

        "max_reserved_gb":
            torch.cuda.max_memory_reserved(i) / 1024**3
    }

print("=" * 70)
print("DISTILLED STUDENT — GPU MEMORY")
print("=" * 70)

for gpu, vals in distilled_memory["gpus"].items():

    print("\n", gpu)

    for k, v in vals.items():
        print(f"{k}: {v:.4f}")

with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_memory.json"
    ),
    "w"
) as f:
    json.dump(
        distilled_memory,
        f,
        indent=4
    )

DISTILLED STUDENT — GPU MEMORY

 gpu_0
allocated_gb: 2.2703
reserved_gb: 6.0723
max_allocated_gb: 2.2711
max_reserved_gb: 6.0723

 gpu_1
allocated_gb: 0.0089
reserved_gb: 2.8730
max_allocated_gb: 0.0089
max_reserved_gb: 2.8730


In [56]:
def get_dir_size_gb(path):

    total = 0

    for root, dirs, files in os.walk(path):
        for file in files:

            fp = os.path.join(root, file)

            total += os.path.getsize(fp)

    return total / (1024**3)


checkpoint_size_gb = get_dir_size_gb(
    DISTILLED_PATH
)

checkpoint_size_mb = (
    checkpoint_size_gb * 1024
)

print(
    "Distilled checkpoint:",
    round(checkpoint_size_mb, 2),
    "MB"
)

print(
    "Distilled checkpoint:",
    round(checkpoint_size_gb, 4),
    "GB"
)

size_result = {
    "model": "distilled_student",
    "parameters": 66961162,
    "checkpoint_size_gb":
        checkpoint_size_gb,
    "checkpoint_size_mb":
        checkpoint_size_mb
}

with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_size.json"
    ),
    "w"
) as f:
    json.dump(
        size_result,
        f,
        indent=4
    )

Distilled checkpoint: 256.13 MB
Distilled checkpoint: 0.2501 GB


In [58]:
!pip install -q codecarbon==3.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.9/396.9 kB 8.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 79.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 90.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.3.0 requires cryptography<49,>=42, but you have cryptography 50.0.1 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 50.0.1 which is incompatible.
pyopenssl 24.2.1 requires cryptography

In [59]:
import codecarbon
print("CodeCarbon:", codecarbon.__version__)

CodeCarbon: 3.3.0


In [60]:
from codecarbon import EmissionsTracker

ENERGY_SAMPLE_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/energy_sample_100.csv"
)

energy_df = pd.read_csv(
    ENERGY_SAMPLE_PATH
)

ENERGY_DIR = os.path.join(
    PHASE10_DIR,
    "energy_logs",
    "distilled_student"
)

os.makedirs(
    ENERGY_DIR,
    exist_ok=True
)

tracker = EmissionsTracker(
    output_dir=ENERGY_DIR,
    output_file="emissions.csv",
    log_level="error"
)

print("Starting energy measurement...")

tracker.start()

start = time.perf_counter()

for i, row in energy_df.iterrows():

    _ = predict_distilled(
        row["text_clean"]
    )

    if (i + 1) % 20 == 0:
        print(f"{i+1}/100")

runtime_energy = time.perf_counter() - start

tracker.stop()

cc = pd.read_csv(
    os.path.join(
        ENERGY_DIR,
        "emissions.csv"
    )
)

last = cc.iloc[-1]

distilled_energy = {
    "model": "distilled_student",
    "n_samples": 100,
    "runtime_s": float(runtime_energy),
    "energy_kwh": float(last["energy_consumed"]),
    "emissions_kg_co2": float(last["emissions"])
}

print("\nDISTILLED STUDENT — ENERGY")
print("=" * 60)

for k, v in distilled_energy.items():
    print(k, ":", v)

with open(
    os.path.join(
        PHASE10_DIR,
        "distilled_student_energy.json"
    ),
    "w"
) as f:
    json.dump(
        distilled_energy,
        f,
        indent=4
    )

[codecarbon WARNING @ 14:14:28] Multiple instances of codecarbon are allowed to run at the same time.


Starting energy measurement...
20/100
40/100
60/100
80/100
100/100

DISTILLED STUDENT — ENERGY
model : distilled_student
n_samples : 100
runtime_s : 0.49651496399746975
energy_kwh : 1.4132845669886998e-05
emissions_kg_co2 : 6.396819562087144e-06


In [61]:
import gc
import torch

# Remove objects that may still hold GPU tensors
for name in [
    "student_model",
    "hard_model",
    "teacher_model",
    "base_model",
    "optimizer",
    "hard_optimizer",
    "outputs",
    "loss",
    "soft_loss",
    "hard_loss",
    "batch",
    "input_ids",
    "attention_mask",
    "hard_labels",
    "soft_labels",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("GPU state after cleanup:")

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 4),
        "GB allocated"
    )

GPU state after cleanup:
GPU 0: 0.0168 GB allocated
GPU 1: 0.0089 GB allocated


In [62]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

DISTILLED_PATH = "/kaggle/working/distilled-student"

device = torch.device("cuda:0")

student_tokenizer = DistilBertTokenizerFast.from_pretrained(
    DISTILLED_PATH
)

student_model = (
    DistilBertForSequenceClassification
    .from_pretrained(DISTILLED_PATH)
    .to(device)
)

student_model.eval()

print("Clean student loaded.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Clean student loaded.


In [63]:
@torch.no_grad()
def predict_distilled(text):

    enc = student_tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding=False,
        return_tensors="pt"
    )

    enc = {
        k: v.to(device)
        for k, v in enc.items()
    }

    logits = student_model(**enc).logits

    pred_idx = logits.argmax(dim=-1).item()

    return CATEGORIES[pred_idx]

In [64]:
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict_distilled(
    benchmark_df["text_clean"].iloc[0]
)

distilled_memory = {
    "model": "distilled_student",
    "gpus": {}
}

for i in range(torch.cuda.device_count()):

    distilled_memory["gpus"][f"gpu_{i}"] = {
        "allocated_gb":
            torch.cuda.memory_allocated(i) / 1024**3,

        "reserved_gb":
            torch.cuda.memory_reserved(i) / 1024**3,

        "max_allocated_gb":
            torch.cuda.max_memory_allocated(i) / 1024**3,

        "max_reserved_gb":
            torch.cuda.max_memory_reserved(i) / 1024**3
    }

print("=" * 70)
print("DISTILLED STUDENT — CLEAN GPU MEMORY")
print("=" * 70)

for gpu, vals in distilled_memory["gpus"].items():

    print("\n", gpu)

    for k, v in vals.items():
        print(f"{k}: {v:.4f}")

import json
import os

with open(
    "/kaggle/working/phase10-results/"
    "distilled_student_memory.json",
    "w"
) as f:
    json.dump(distilled_memory, f, indent=4)

DISTILLED STUDENT — CLEAN GPU MEMORY

 gpu_0
allocated_gb: 0.2672
reserved_gb: 4.0859
max_allocated_gb: 0.2681
max_reserved_gb: 4.0859

 gpu_1
allocated_gb: 0.0089
reserved_gb: 2.8730
max_allocated_gb: 0.0089
max_reserved_gb: 2.8730


In [65]:
def get_dir_size_gb(path):

    total_bytes = 0

    for root, dirs, files in os.walk(path):
        for filename in files:
            fp = os.path.join(root, filename)
            total_bytes += os.path.getsize(fp)

    return total_bytes / (1024**3)


size_gb = get_dir_size_gb(
    "/kaggle/working/distilled-student"
)

size_mb = size_gb * 1024

print("Distilled checkpoint size:")
print("MB:", round(size_mb, 2))
print("GB:", round(size_gb, 4))

size_results = {
    "model": "distilled_student",
    "parameters": 66961162,
    "checkpoint_size_mb": size_mb,
    "checkpoint_size_gb": size_gb
}

with open(
    "/kaggle/working/phase10-results/"
    "distilled_student_size.json",
    "w"
) as f:
    json.dump(size_results, f, indent=4)

Distilled checkpoint size:
MB: 256.13
GB: 0.2501


In [66]:
import pandas as pd

phase10_metrics = pd.DataFrame([
    {
        "model": "distilled_student",
        "parameters": 66961162,
        "accuracy": 0.7735,
        "macro_precision": 0.7600140069690334,
        "macro_recall": 0.7680111060543814,
        "macro_f1": 0.7629680730982821,
        "weighted_f1": 0.7715770921582532,
        "benchmark_runtime_s": 9.061206254002172,
        "benchmark_throughput_sps": 220.7211649240007,
        "formal_mean_latency_s": 0.004398239360307343,
        "formal_throughput_sps": 227.36370581025434,
        "energy_100_kwh": 1.4132845669886998e-05,
        "co2_100_kg": 6.396819562087144e-06,
        "checkpoint_size_gb": size_gb,
        "unmatched": 0
    },
    {
        "model": "hard_label_student",
        "parameters": 66961162,
        "accuracy": 0.764,
        "macro_precision": 0.7538058234843132,
        "macro_recall": 0.7479424017301597,
        "macro_f1": 0.7466958537197546,
        "weighted_f1": 0.7551154514211491,
        "benchmark_runtime_s": 8.951577346000704,
        "benchmark_throughput_sps": 223.4243109001946,
        "unmatched": 0
    }
])

phase10_metrics.to_csv(
    "/kaggle/working/phase10-results/phase10_metrics.csv",
    index=False
)

print(phase10_metrics)

                model  parameters  accuracy  macro_precision  macro_recall  \
0   distilled_student    66961162    0.7735         0.760014      0.768011   
1  hard_label_student    66961162    0.7640         0.753806      0.747942   

   macro_f1  weighted_f1  benchmark_runtime_s  benchmark_throughput_sps  \
0  0.762968     0.771577             9.061206                220.721165   
1  0.746696     0.755115             8.951577                223.424311   

   formal_mean_latency_s  formal_throughput_sps  energy_100_kwh  co2_100_kg  \
0               0.004398             227.363706        0.000014    0.000006   
1                    NaN                    NaN             NaN         NaN   

   checkpoint_size_gb  unmatched  
0            0.250126          0  
1                 NaN          0  


In [67]:
import os
import shutil

BACKUP_DIR = "/kaggle/working/phase10-complete-backup"

if os.path.exists(BACKUP_DIR):
    shutil.rmtree(BACKUP_DIR)

os.makedirs(BACKUP_DIR)

# All numerical/result artifacts
shutil.copytree(
    "/kaggle/working/phase10-results",
    os.path.join(
        BACKUP_DIR,
        "phase10-results"
    )
)

# Distilled student — important for Phase 11+
shutil.copytree(
    "/kaggle/working/distilled-student",
    os.path.join(
        BACKUP_DIR,
        "distilled-student"
    )
)

# Hard-label control
if os.path.exists(
    "/kaggle/working/hard-label-student"
):
    shutil.copytree(
        "/kaggle/working/hard-label-student",
        os.path.join(
            BACKUP_DIR,
            "hard-label-student"
        )
    )

# Exact benchmark
shutil.copy(
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv",
    os.path.join(
        BACKUP_DIR,
        "benchmark_2000.csv"
    )
)

ZIP_BASE = (
    "/kaggle/working/"
    "phase10-final-backup-COMPLETE"
)

shutil.make_archive(
    ZIP_BASE,
    "zip",
    BACKUP_DIR
)

zip_path = ZIP_BASE + ".zip"

print("=" * 70)
print("PHASE 10 COMPLETE")
print("=" * 70)
print("ZIP:", zip_path)

print(
    "ZIP size:",
    round(
        os.path.getsize(zip_path) / 1024**2,
        2
    ),
    "MB"
)

PHASE 10 COMPLETE
ZIP: /kaggle/working/phase10-final-backup-COMPLETE.zip
ZIP size: 474.34 MB
